<a href="https://colab.research.google.com/github/KarthikRed2000/grounded_vla/blob/main/colab/03_run_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03: Smoke Test + Baselines + ORA

Workhorse notebook. Runs `grounded-vla smoke`, then evaluates the three agents
(ReAct + Mistral, single-shot LLaVA, ORA + LLaVA) across ScienceQA, the
synthetic sample, and Mind2Web. All artifacts go to
`/content/drive/MyDrive/grounded_vla/runs/<agent>__<dataset>/`.

**Runtime:** A100 GPU **strongly recommended** (Pro+ default).

**Estimated wallclock:** ~60-90 min on A100 for the full sweep.
**Estimated compute units:** ~15-20 (well within Pro+'s 500/month).

## Pro+ tip: background execution

Tools → Settings → check **'Run after disconnecting'**. With this on, you can
close the tab; Colab keeps the runtime alive for up to 24 hours and the
checkpointed `EvalRunner` will keep writing per-task results to Drive. Re-open
the notebook later and the results will be waiting in Drive.

If a session does die mid-sweep, just re-run — `resume=True` skips already-
completed tasks via the trajectory JSONs already on Drive.

In [1]:
# Mount Google Drive. The first run prompts for OAuth; subsequent runs
# in the same session reuse the token automatically.
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/grounded_vla')
ROOT.mkdir(parents=True, exist_ok=True)
print('drive root:', ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
drive root: /content/drive/MyDrive/grounded_vla


In [2]:
# Verify accelerator. Pro+ should give you A100 most of the time; if you
# see T4, change Runtime -> Change runtime type -> A100 GPU and re-run.
import subprocess
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                '--format=csv'])

CompletedProcess(args=['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], returncode=0)

In [3]:
# Clone repo if absent, then always pull to get the latest fixes.
# With an editable install (-e) a pull is all that's needed — no reinstall.
REPO_URL = 'https://github.com/KarthikRed2000/grounded_vla.git'
import os, subprocess
if not os.path.exists('/content/grounded_vla'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/grounded_vla'], check=True)
subprocess.run(['git', '-C', '/content/grounded_vla', 'pull'], check=True)
%cd /content/grounded_vla

/content/grounded_vla


In [4]:
# Install repo + GPU stack. Quiet to keep the cell output sane.
!pip install -q -e .[gpu,data]

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for grounded_vla (pyproject.toml) ... done


In [5]:
# Point HF cache at the Drive-mounted weights so transformers loads from there.
import os
WEIGHTS = '/content/drive/MyDrive/grounded_vla/hf_cache'
DATA    = '/content/drive/MyDrive/grounded_vla/data'
os.environ['HF_HOME'] = WEIGHTS
os.environ['TRANSFORMERS_CACHE'] = WEIGHTS
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
assert os.path.isdir(WEIGHTS), 'run notebook 01 first'
assert os.path.isdir(DATA),    'run notebook 02 first'

In [6]:
# Smoke test: pure-mock pipeline. Should print 'smoke ok' in <5 seconds.
!grounded-vla smoke

smoke ok: agent=ora-vlm, n=46, success=0.00, mean_steps=1.50


In [7]:
# Build the three agents. On A100 we can use the default GenerationConfig
# without any concessions to memory.
from grounded_vla.agents import ORAAgent, ReActAgent, SingleShotVLMAgent
from grounded_vla.backends import make_backend
from grounded_vla.backends.base import GenerationConfig

WEIGHTS = '/content/drive/MyDrive/grounded_vla/hf_cache'

def build_react():
    backend = make_backend({
        'kind': 'mistral',
        'model_id': f'{WEIGHTS}/Mistral-7B-Instruct-v0.2',
        'device': 'cuda',
        'quantize': '4bit',
    })
    return ReActAgent(backend, GenerationConfig(max_new_tokens=384, temperature=0.1))

def build_single_shot():
    backend = make_backend({
        'kind': 'llava',
        'model_id': f'{WEIGHTS}/llava-v1.6-mistral-7b-hf',
        'device': 'cuda',
        'quantize': '4bit',
    })
    return SingleShotVLMAgent(backend, GenerationConfig(max_new_tokens=256, temperature=0.0))

def build_ora():
    backend = make_backend({
        'kind': 'llava',
        'model_id': f'{WEIGHTS}/llava-v1.6-mistral-7b-hf',
        'device': 'cuda',
        'quantize': '4bit',
    })
    return ORAAgent(backend, GenerationConfig(max_new_tokens=384, temperature=0.1))

In [8]:
import sys, importlib

# Nuke every cached grounded_vla module entry
to_drop = [m for m in sys.modules if m == 'grounded_vla' or m.startswith('grounded_vla.')]
for m in to_drop:
    del sys.modules[m]
print(f"dropped {len(to_drop)} cached entries")

importlib.invalidate_caches()

# Force a fresh import
from grounded_vla.data import make_dataset
print("OK:", make_dataset)

dropped 17 cached entries
OK: <function make_dataset at 0x7bb7021da160>


In [15]:
# Build the three datasets pointing at the Drive-mounted data.
from grounded_vla.data import make_dataset
DATA = '/content/drive/MyDrive/grounded_vla/data'

def scienceqa(limit=50):
    return make_dataset({
        'kind': 'scienceqa',
        'jsonl_path': f'{DATA}/scienceqa/test.jsonl',
        'images_dir': f'{DATA}/scienceqa',
        'limit': limit,
    })

def _mind2web_jsonl():
    # Auto-discover whichever split notebook 02 produced (test_task,
    # test_website, train fallback, ...).
    import glob
    candidates = sorted(glob.glob(f'{DATA}/mind2web/*.jsonl'))
    if not candidates:
        raise FileNotFoundError(f'no Mind2Web JSONL under {DATA}/mind2web/')
    return candidates[0]

def mind2web(limit=30):
    import glob
    candidates = sorted(glob.glob(f'{DATA}/mind2web/*.jsonl'))
    if not candidates:
        raise FileNotFoundError(f'no Mind2Web JSONL under {DATA}/mind2web/')
    return make_dataset({
        'kind': 'mind2web',
        'jsonl_path': candidates[0],
        'images_dir': f'{DATA}/mind2web',   # was wrongly f'{DATA}/mind2web/images'
        'limit': limit,
    })


def synthetic():
    return make_dataset({
        'kind': 'jsonl',
        'path': f'{DATA}/synthetic_sample/synthetic_sample.jsonl',
        'source': 'synthetic',
    })

In [10]:
# Generic eval runner with checkpointing -> Drive. Resumable across sessions.
from grounded_vla.eval import EvalRunner
from pathlib import Path

RUNS = Path('/content/drive/MyDrive/grounded_vla/runs')
RUNS.mkdir(parents=True, exist_ok=True)

def run(agent_name, agent, ds_name, dataset, **kw):
    out = RUNS / f'{agent_name}__{ds_name}'
    runner = EvalRunner(agent)
    res = runner.evaluate(
        dataset, save_dir=out, checkpoint_every=5, resume=True, **kw
    )
    print(f'{agent_name} on {ds_name}: '
          f'success={res.task_completion_rate:.3f} '
          f'mean_steps={res.mean_steps:.2f} '
          f'errors={res.error_breakdown}')
    return res

## ReAct + Mistral (Baseline 1)

Text-only. ~1-2 sec/task on A100 with 4-bit Mistral. ~3 minutes total.

In [20]:
# import shutil, pathlib
# root = pathlib.Path("/content/drive/MyDrive/grounded_vla/runs")
# for name in ("ora_llava__scienceqa", "ora_llava__synthetic", "ora_llava__mind2web"):
#     d = root / name
#     if d.exists():
#         shutil.rmtree(d)
#         print("cleared", d)


cleared /content/drive/MyDrive/grounded_vla/runs/ora_llava__scienceqa
cleared /content/drive/MyDrive/grounded_vla/runs/ora_llava__synthetic
cleared /content/drive/MyDrive/grounded_vla/runs/ora_llava__mind2web


In [12]:
import subprocess, sys
subprocess.run(['git', '-C', '/content/grounded_vla', 'pull'], check=True)
for m in list(sys.modules):
    if m.startswith('grounded_vla'):
        del sys.modules[m]
print("ready")

ready


In [13]:
# 1. Pull the fix
import subprocess
subprocess.run(['git', '-C', '/content/grounded_vla', 'pull'], check=True)
print("pulled")

# 2. Force Python to reload the updated schemas module
import importlib, sys
for m in list(sys.modules):
    if m.startswith('grounded_vla'):
        del sys.modules[m]
importlib.invalidate_caches()
print("cache cleared")

# 3. Wipe corrupted checkpoints
# import shutil, pathlib
# root = pathlib.Path("/content/drive/MyDrive/grounded_vla/runs")
# for name in ("react_mistral__scienceqa", "react_mistral__synthetic", "react_mistral__mind2web"):
#     d = root / name
#     if d.exists():
#         shutil.rmtree(d)
#         print("cleared", d)

pulled
cache cleared


In [16]:
react = build_react()
_ = run('react_mistral', react, 'scienceqa',  scienceqa(limit=50))
_ = run('react_mistral', react, 'synthetic',  synthetic())
_ = run('react_mistral', react, 'mind2web',   mind2web(limit=30))
react.backend.close()

[05/05/26 01:30:50] INFO     INFO:grounded_vla.eval.runner:resuming: 50 tasks already on disk

react_mistral on scienceqa: success=0.180 mean_steps=1.12 errors={'reasoning_error': 34, 'action_parsing_failure': 7, 'none': 9}


                    INFO     INFO:grounded_vla.eval.runner:resuming: 3 tasks already on disk

react_mistral on synthetic: success=0.000 mean_steps=1.33 errors={'reasoning_error': 2, 'action_parsing_failure': 1}


                    INFO     INFO:grounded_vla.eval.runner:resuming: 3 tasks already on disk

react_mistral on mind2web: success=0.000 mean_steps=1.47 errors={'action_parsing_failure': 30}


## Single-shot LLaVA (Baseline 2)

VLM, one call per task. Tests H1 (visual grounding helps).

In [21]:
import subprocess, importlib, sys, shutil

# 1. Pull the fix
subprocess.run(['git', '-C', '/content/grounded_vla', 'pull'], check=True)

# 2. Reload the data loader (so the fix is live without restart)
for key in [k for k in sys.modules if k.startswith('grounded_vla.data')]:
    del sys.modules[key]
import grounded_vla.data.base  # force re-import

# 3. Clear old bad checkpoint
# shutil.rmtree('/content/drive/MyDrive/grounded_vla/runs/single_shot_llava__mind2web', ignore_errors=True)
print("Ready")

Ready


In [18]:
ss = build_single_shot()
_ = run('single_shot_llava', ss, 'scienceqa',  scienceqa(limit=50))
_ = run('single_shot_llava', ss, 'synthetic',  synthetic())
_ = run('single_shot_llava', ss, 'mind2web',   mind2web(limit=30))
ss.backend.close()

[05/05/26 01:32:00] INFO     INFO:numexpr.utils:NumExpr defaulting to 12 threads.

The image processor of type `LlavaNextImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[05/05/26 01:33:09] INFO     INFO:grounded_vla.eval.runner:checkpoint @ task 5 ->                                  
                             /content/drive/MyDrive/grounded_vla/runs/single_shot_llava__scienceqa

...

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[05/05/26 01:43:10] INFO     INFO:grounded_vla.eval.runner:checkpoint @ task 25 ->                                 
                             /content/drive/MyDrive/grounded_vla/runs/single_shot_llava__mind2web

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[05/05/26 01:43:55] INFO     INFO:grounded_vla.eval.runner:checkpoint @ task 30 ->                                 
                             /content/drive/MyDrive/grounded_vla/runs/single_shot_llava__mind2web

single_shot_llava on mind2web: success=0.000 mean_steps=1.00 errors={'visual_misgrounding': 21, 'reasoning_error': 9}


## ORA + LLaVA (Our Method)

VLM with the Observe-Reason-Act loop and per-step visual re-encoding. Tests H2.
This is the single longest cell in the project (~30-50 min on A100).

In [22]:
ora = build_ora()
_ = run('ora_llava', ora, 'scienceqa',  scienceqa(limit=50))
_ = run('ora_llava', ora, 'synthetic',  synthetic())
_ = run('ora_llava', ora, 'mind2web',   mind2web(limit=30))
ora.backend.close()

Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[05/05/26 01:47:13] INFO     INFO:grounded_vla.eval.runner:checkpoint @ task 5 ->                                  
                             /content/drive/MyDrive/grounded_vla/runs/ora_llava__scienceqa

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[05/05/26 01:48:38] INFO     INFO:grounded_vla.eval.runner:checkpoint @ task 10 ->                                 
                             /content/drive/MyDrive/grounded_vla/runs/ora_llava__scienceqa

...

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

[05/05/26 03:11:29] INFO     INFO:grounded_vla.eval.runner:checkpoint @ task 25 ->                                 
                             /content/drive/MyDrive/grounded_vla/runs/ora_llava__mind2web

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

[05/05/26 03:26:41] INFO     INFO:grounded_vla.eval.runner:checkpoint @ task 30 ->                                 
                             /content/drive/MyDrive/grounded_vla/runs/ora_llava__mind2web

ora_llava on mind2web: success=0.000 mean_steps=20.00 errors={'truncated': 30}


In [23]:
# Aggregate results table for the report.
import json
from pathlib import Path
rows = []
for d in sorted(Path('/content/drive/MyDrive/grounded_vla/runs').iterdir()):
    s = json.loads((d / 'summary.json').read_text())
    rows.append((d.name, s['n_tasks'], s['task_completion_rate'], s['mean_steps'], s['error_breakdown']))
for r in rows:
    print(f'{r[0]:30s}  n={r[1]:3d}  success={r[2]:.3f}  steps={r[3]:.2f}  errors={r[4]}')

ora_llava__mind2web             n= 30  success=0.000  steps=20.00  errors={'truncated': 30}
ora_llava__scienceqa            n= 50  success=0.600  steps=1.22  errors={'none': 30, 'visual_misgrounding': 15, 'reasoning_error': 3, 'truncated': 2}
ora_llava__synthetic            n=  3  success=1.000  steps=1.00  errors={'none': 3}
react_mistral__mind2web         n= 30  success=0.000  steps=1.47  errors={'action_parsing_failure': 30}
react_mistral__scienceqa        n= 50  success=0.180  steps=1.12  errors={'reasoning_error': 34, 'action_parsing_failure': 7, 'none': 9}
react_mistral__synthetic        n=  3  success=0.000  steps=1.33  errors={'reasoning_error': 2, 'action_parsing_failure': 1}
single_shot_llava__mind2web     n= 30  success=0.000  steps=1.00  errors={'visual_misgrounding': 21, 'reasoning_error': 9}
single_shot_llava__scienceqa    n= 50  success=0.480  steps=1.00  errors={'none': 24, 'visual_misgrounding': 15, 'reasoning_error': 8, 'action_parsing_failure': 3}
single_shot_llava__

## Done

Results live at `/content/drive/MyDrive/grounded_vla/runs/`. The H1 and H2
comparisons drop straight out of the table above:

- **H1 (vision helps):** compare `react_mistral` vs `single_shot_llava` per dataset.
- **H2 (loop helps):** compare `single_shot_llava` vs `ora_llava` per dataset.

Move on to `04_lora_finetune.ipynb` for the H3 stretch goal.